# 04 · Fields and samplers

A **sampler** maps events x = (a, x, y, z) (shape (..., 4), comoving Mpc) to the value of a gas field there. The optical
depth needs, for every absorbing species, five fields (review notes, Sec. GasProperties):

| field | unit | kind |
|---|---|---|
| `temperature` | K | scalar |
| `velocity` | km/s (peculiar) | vector (…, 3) |
| `number_density` | comoving cm⁻³ | scalar |
| `turbulence` | m/s (b_turb) | scalar |
| `collisional` | GHz (A_coll) | scalar |

`ScalarField` / `VectorField` wrap any callable; `Fields({species: {name: field}})` holds them all. Samplers shipped
with the core: `LogRamp`, `VelocityRamp`, `Constant`, `GaussianDensity` (a `PowerLawGRF` realisation) and
`SampledField` (values already known at the points, e.g. from a simulation). All are JAX pytrees. Toy samplers first (ramps, a Gaussian random field), then fields known at the points. Runtime: ~1 min.

**Requires:** nothing: samplers are functions of events (a, x, y, z); `SampledField` holds values laid out on the points of a skewer bundle (03, 05).  
**Independent of:** lines, cosmology.

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)          # SkewerJAX follows this flag; it never sets it itself
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import (use_style, use_backend, ENTITY_COLORS as EC, MODE_STYLES as MS, PALETTE as TOL, FG, BG, DIM,
                                      CMAPS, LABELS, FIGSIZES, add_panel_letters, plot_dag, plot_spectra, zoom_inset)
# use_backend()      # interactive figures (ipympl) in a live notebook; inline when executed headless or without ipympl
use_style()        # dark style: Tol's colour-blind-safe "bright" colours and colour maps, minimum font sizes

In [ ]:
plot_dag(('samplers', 'fields',)); plt.show()

In [ ]:
from skewerjax.core import (Fields, ScalarField, VectorField, SampledField, LogRamp, VelocityRamp, Constant, PowerLawGRF,
                            GaussianDensity, plane_points)
L = 1.0
species = {"temperature": ScalarField(LogRamp(1e4, 1e6, L, axis=3)),          # log ramp along z
           "velocity": VectorField(VelocityRamp(-50.0, 80.0, L)),             # v_x linear in x
           "number_density": ScalarField(LogRamp(1e-9, 1e-3, L, axis=2)),     # log ramp along y
           "turbulence": ScalarField(Constant(0.0)), "collisional": ScalarField(Constant(0.0))}
fields = Fields({"HI": species, "SiIII": species})
pts = plane_points(L, "x", 0.5, 64, 64)                                        # (64, 64, 3) on the plane x = 0.5
events = jnp.concatenate([jnp.full(pts.shape[:-1] + (1,), 0.9), pts], -1)       # a = 0.9
directions = jnp.broadcast_to(jnp.array([1.0, 0.0, 0.0]), pts.shape)            # photons along +x
sampled = fields.sample(events, directions)
print({sp: {k: v.shape for k, v in d.items()} for sp, d in sampled.items()})

`sample` adds `LoS_velocity`, the projection of the velocity on the photon direction. Across species, `fields.temperature(x)` stacks them: shape (N_species, …).

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(20, 6), layout="constrained")
ext = (0, L, 0, L)
for ax, (name, cmap) in zip(axs[:2], (("temperature", "incandescent"), ("number_density", "iridescent"))):
    from skewerjax.utils.plotting import tol_cmap
    im = ax.imshow(np.log10(np.asarray(sampled["HI"][name])).T, origin="lower", extent=ext, cmap=tol_cmap(cmap))
    fig.colorbar(im, ax=ax, label=f"log10 {name}")
    ax.set(xlabel="y [cMpc]", ylabel="z [cMpc]", title=f"LogRamp: {name}")
xs = jnp.linspace(0, L, 50)
ev_x = jnp.stack([jnp.full(50, 0.9), xs, jnp.full(50, 0.5), jnp.full(50, 0.5)], -1)
axs[2].plot(xs, fields.LoS_velocity(ev_x, jnp.broadcast_to(jnp.array([1.0, 0, 0]), (50, 3)))[0], color=EC["velocity"])
axs[2].set(xlabel="x [cMpc]", ylabel=r"$v_\parallel$ [km/s]", title="VelocityRamp along x, projected on +x")
add_panel_letters(axs); plt.show()

## 1 · Gaussian random fields: `PowerLawGRF`

P(k) = A_s (k/k_0)^{n_s−1} on a periodic grid in 1, 2 or 3 dimensions, given by (L, N) or by two of (k_min, k_max, N_k).
Normalisation as the P1D estimator of the review notes: ⟨|δ_k|²⟩/V = P(k), so `variance()` is the expected σ².
`sample(key)` is jitted; `jax.vmap` over keys draws a batch.

In [ ]:
key = jax.random.PRNGKey(1)
g1 = PowerLawGRF(A_s=1.0, k_0=1.0, n_s=-0.5, L=100.0, N=1024)
g2 = PowerLawGRF(A_s=1.0, k_0=1.0, n_s=-1.5, L=(50.0, 100.0), N=(128, 256))     # anisotropic box: y = 2 x
g3 = PowerLawGRF(A_s=4.0, k_0=2.0, n_s=-1.25, k_min=(2 * np.pi / 50,) * 3, N_k=(256,) * 3)   # mode (b): N_k modes per axis
for g in (g1, g2, g3):
    print(g, f"sigma = {float(jnp.sqrt(g.variance())):.3f}")
d1, d2, d3 = g1.sample(key), g2.sample(key), g3.sample(key)
iz = d3.shape[2] // 2                                                              # the slice z = L/2
fig, axs = plt.subplots(1, 3, figsize=(24, 7.5), layout="constrained", gridspec_kw={"width_ratios": [1.3, 0.75, 1]})
axs[0].plot(g1.x_grid()[0], d1, color=EC["density"], lw=1); axs[0].set(xlabel="x [Mpc]", ylabel=r"$\delta$", title="1D, $n_s = -0.5$")
Lx, Ly = g2.L
im = axs[1].imshow(np.asarray(d2).T, origin="lower", extent=(0, Lx, 0, Ly), cmap=CMAPS["diverging"], aspect="equal")
axs[1].set(xlabel="x [Mpc]", ylabel="y [Mpc]", title="2D, box 50 × 100 Mpc")
L3 = g3.L[0]
axs[2].imshow(np.asarray(d3[:, :, iz]).T, origin="lower", extent=(0, L3, 0, L3), cmap=CMAPS["diverging"], aspect="equal")
axs[2].set(xlabel="x [Mpc]", ylabel="y [Mpc]", title=f"3D, {d3.shape[0]}³ cells, slice z = L/2 (index {iz})")
add_panel_letters(axs); plt.show()

The measured power spectrum of many draws recovers the input in every dimension. The estimator is the one that
defines the normalisation, P̂(k) = |δ_k|²/V (δ_k the discrete Fourier transform times the cell volume), averaged
over the modes of each |k| bin and over the draws (error bars: the scatter between draws).
- **Isotropic P(|k|)** (top rows): every mode in a bin of |k|, in 1D, 2D and 3D, against the input, with residuals.
- **Parallel and perpendicular modes** (bottom rows, 2D and 3D): the modes along the last axis (k_⊥ = 0) and across
  it (k_∥ = 0). For an isotropic P(k) both follow the same input curve: that is what allows the isotropic average above.
  In the anisotropic 2D box (y = 2x) the two directions have different fundamental modes, 2π/100 and 2π/50 Mpc⁻¹, and
  still follow one P(k).

Full checks: `scripts/validation/check_summary_statistics.py`.

In [ ]:
from skewerjax.utils.plotting import plot_pk_recovery_nd
draws = lambda g, n, seed: np.asarray(jax.vmap(g.sample)(jax.random.split(jax.random.PRNGKey(seed), n)))
cases = {"1D, 200 draws": (g1, draws(g1, 200, 2)), "2D (50 × 100 Mpc), 40 draws": (g2, draws(g2, 40, 3)),
         "3D (256³), 4 draws": (g3, draws(g3, 4, 4))}
fig, axs, rec = plot_pk_recovery_nd(cases)
add_panel_letters(axs[:2].ravel()); fig.suptitle("PowerLawGRF: the measured P(k) is the input, isotropically and along each direction", fontsize=22)
plt.show()
for name, r in rec.items():
    print(f"{name:28s}: isotropic P(|k|), chi2/dof = {r['chi2_dof']:.2f}, max |P / P_in - 1| = {r['max_rel']:.3f}")

## 2 · Densities from a GRF, and fields known at the points

**`GaussianDensity(key, grf, n0_min, n0_max, contrast)`.** δ is one `PowerLawGRF` realisation divided by its expected
σ, interpolated trilinearly in the periodic box, and mapped to

$$\log_{10} n = \tfrac12(\log_{10} n_{0,\min} + \log_{10} n_{0,\max}) + \tfrac12(\log_{10} n_{0,\max} - \log_{10} n_{0,\min})\,\tanh(\delta/\mathrm{contrast}) .$$

- `n0_min` and `n0_max` are the asymptotes of the tanh: they set the centre and the half-width (in dex) of log n.
- The interpolation is linear, so δ at the points is still Gaussian (smoothed near the grid scale).
- **log n is not Gaussian**: a tanh of a Gaussian is bounded (excess kurtosis −1.37 for contrast 1, −0.37 for 4).
- **n is neither Gaussian nor lognormal.** Only for a large contrast does log n tend to a Gaussian (n lognormal).
- So this sampler is a *bounded* density with a GRF inside, not a Gaussian density. The Gaussian statistics live in
  `PowerLawGRF`, whose δ can be used directly (or as n = n̄(1 + δ)) when a Gaussian density is needed.

`SampledField(values)` returns values given at the points of a `SkewerBundle` (shape (N_sk, N_ppsk)): that is how
simulation fields enter (05).

In [ ]:
dens = GaussianDensity(jax.random.PRNGKey(0), g3, n0_min=1e-12, n0_max=1e-9, contrast=1.2)
y = jnp.linspace(0, 50, 2000)
ev = jnp.stack([jnp.ones_like(y), jnp.full_like(y, 25.0), y, jnp.full_like(y, 10.0)], -1)
values = SampledField(dens(ev)[None])                     # the same values, now "known at the points"
fig, ax = plt.subplots(figsize=(14, 5), layout="constrained")
ax.semilogy(y, dens(ev), color=EC["density"], label="GaussianDensity (interpolated)")
ax.semilogy(y, values(ev), color=FG, **MS["matched"], label="SampledField of the same values")
ax.set(xlabel="y [cMpc]", ylabel=r"$n\ [\mathrm{cm^{-3}}]$", title="A density skewer through the 3D GRF"); ax.legend(); plt.show()

## 3 · Fields are pytrees

Samplers built from these classes pass through `jax.jit` and `jax.vmap` with the arrays they hold as leaves (plain
Python callables are kept as static data, so closures work too).

In [ ]:
f_jit = jax.jit(lambda f, x, d: f.sample(x, d))
out = f_jit(fields, events, directions)
diff = max(float(jnp.max(jnp.abs(out[sp][k] - sampled[sp][k]) / jnp.maximum(jnp.abs(sampled[sp][k]), 1e-300))) for sp in sampled for k in sampled[sp])
print(f"jit(sample) vs sample: max relative difference {diff:.1e} (rounding) | leaves of the Fields pytree: {len(jax.tree_util.tree_leaves(fields))}")